In [2]:
import pandas as pd

In [3]:
import pandas as pd
import re

# Input Files
sample_inventory_file = "AlcHepNet_Sample_Inventory_2025-04-21.xlsx"
case_obs_file = "case_obs_DCC_data_release_v2-0-4.tsv"
case_rct_file = "case_rct_DCC_data_release_v2-0-4.tsv"
output_file = "aliquot_DCC_data_release_v2-0-5_inventory.tsv"
qc_file = "QC-no-follow-up_aliquot_DCC_data_release_v2-0-5_inventory_updated-labs.tsv"

# Read datasets
sample_df = pd.read_excel(sample_inventory_file, dtype=str)
case_obs_df = pd.read_csv(case_obs_file, sep="\t", dtype=str)
case_rct_df = pd.read_csv(case_rct_file, sep="\t", dtype=str)

# Extract External Subject ID and label cohort type
def extract_subject_map(df, cohort):
    df["External Subject ID"] = df["*submitter_id"].str.extract(r"^(\d+)")
    return df[["External Subject ID", "*submitter_id"]].rename(
        columns={"*submitter_id": "case_submitter_id"}
    ).assign(cohort=cohort)

obs_map = extract_subject_map(case_obs_df, "obs")
rct_map = extract_subject_map(case_rct_df, "clinical")
subject_map = pd.concat([obs_map, rct_map], ignore_index=True).drop_duplicates("External Subject ID")

# Merge sample inventory with subject mapping
merged = sample_df.merge(subject_map, on="External Subject ID", how="left")

# Function to generate *follow_ups.submitter_id
def generate_followup_id(row):
    patient_id = row["External Subject ID"]
    event_label = row.get("Event Label", "")
    
    if not isinstance(patient_id, str) or not isinstance(event_label, str):
        return None
    
    event_label = event_label.strip()

    if event_label.startswith("Week"):
        match = re.search(r"Week\s*(\d+)", event_label)
        if match:
            week_num = int(match.group(1))
            day_equivalent = week_num * 7
            return f"{patient_id}_obs_{day_equivalent}"
    elif event_label.startswith("Day"):
        match = re.search(r"Day\s*(\d+)", event_label)
        if match:
            day_num = int(match.group(1))
            return f"{patient_id}_clinical_{day_num}"
    
    return None

# Apply follow-up ID generation
merged["*follow_ups.submitter_id"] = merged.apply(generate_followup_id, axis=1)

# Standardize specimen_type
specimen_mapping = {
    "CPT plasma": "CPT Plasma",
    "HCl sodium citrate plasma": "HCl Sodium Citrate Plasma",
    "HCl acidified plasma": "HCl Acidified Plasma",
    "Neat plasma": "NEAT Plasma",
    "Platelet poor plasma": "Platelet Poor Plasma",
    "Platelet rich plasma": "Platelet Rich Plasma",
    "Whole blood": "Whole Blood (DNA)",
    "Liver biopsy": "Liver Tissue",
    "Serum for Anakinra trough level": "Anakinra"
}

merged["specimen_type"] = merged["Type"].replace(specimen_mapping)

# Build the output DataFrame
aliquot_df = pd.DataFrame({
    "*type": ["aliquot"] * len(merged),
    "project_id": ["ARDaC-AlcHepNet"] * len(merged),
    "*submitter_id": merged["Specimen Label"],
    "*follow_ups.submitter_id": merged["*follow_ups.submitter_id"],
    "labs.submitter_id": merged["Visit Site"],
    "aliquot_amount": pd.NA,
    "aliquot_collection_unit": pd.NA,
    "container_type": pd.NA,
    "specimen_type": merged["specimen_type"]
})

# QC: Separate rows with missing follow-up ID
qc_df = aliquot_df[aliquot_df["*follow_ups.submitter_id"].isna()]
valid_df = aliquot_df[aliquot_df["*follow_ups.submitter_id"].notna()]

# Save output files
valid_df.to_csv(output_file, sep="\t", index=False)
qc_df.to_csv(qc_file, sep="\t", index=False)

print(f"Main output written to: {output_file}")
print(f"QC output (missing follow-up ID) written to: {qc_file}")

Main output written to: aliquot_DCC_data_release_v2-0-5_inventory.tsv
QC output (missing follow-up ID) written to: QC-no-follow-up_aliquot_DCC_data_release_v2-0-5_inventory.tsv
